# for Loops and while Loops

**Topic:** `2.2` &nbsp;|&nbsp; **Module 2: Control Flow and Loops**

*Iteration as a protocol: a for loop asks the sequence for the next element, a while loop asks a question - and only one of the two can stop on its own.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M2)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [02_control_flow/README.md](../README.md)

## 1. Why This Topic Matters

Telemetry, routes and retry policies are all loops, and the difference between a loop that stops and a loop that hangs is the difference between a robot that degrades and a robot that is stuck holding a live process. Choosing the right loop is a safety decision, not a style preference.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Explain why a for loop cannot run away over a finite sequence.
2. Give every while loop a bound, a break, or visible progress.
3. Implement __iter__ and __next__ to make an object loopable.
4. Explain why an exhausted iterator stays exhausted.
5. Choose direct iteration over index iteration and say why.
6. Express a retry policy as a bounded for loop.

## 3. Prerequisites

This topic assumes you already have:

- Topic 2.1 Conditional Statements
- Topic 1.8 Writing and Running Your First Scripts

## 4. Mental Model

**Mental model: the for loop asks the sequence for permission, the while loop asks the world a question.** A sequence can say 'no more' - that is what exhaustion means - so a for loop always terminates. A question has no such guarantee, so a while loop terminates only if something inside the body changes the answer.

|  | `for` | `while` |
| --- | --- | --- |
| Who decides to stop | the sequence, by being exhausted | the condition, by becoming false |
| Can it run forever | only if the sequence is infinite | yes, easily |
| Typical bug | iterating a drained iterator | a body that never changes the condition |
| Safe form | `for x in items` | `while ... : ... ; state changes` |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Loop | A block that runs once per iteration. |
| Iteration | One pass through a loop body. |
| Sequence | An object yielding elements one at a time, such as a list. |
| Iterable | An object with an __iter__ method. |
| Iterator | An object with __next__, which raises StopIteration when exhausted. |
| Exhaustion | The state of an iterator after it has yielded everything. |
| StopIteration | The exception that ends a for loop. |
| Infinite loop | A loop that never terminates, usually a while with no progress. |
| Bound | A maximum count that guarantees a retry loop terminates. |
| Accumulator | A variable updated on each pass to hold a running total. |

## 6. Conceptual Explanation

A `for` loop does not know how many iterations it will run. It asks the object you gave it for one element at a time until the object is exhausted. That is why a for loop is safe by construction: there is no counter to get wrong and no condition to forget, so it cannot run forever.

A `while` loop is the opposite. It knows nothing about how many iterations it will run; it re-evaluates a condition until the condition becomes false. That flexibility is exactly why a while loop can run forever: if the condition never becomes false, nothing stops it. Every while loop in production code therefore needs a bound, a timeout, or a state that provably changes.

In [ ]:
waypoints = [(0.0, 0.0), (3.0, 0.0), (3.0, 4.0)]

# for: the sequence decides when to stop
for index, (x, y) in enumerate(waypoints):
    print(f'  visiting {index}: ({x}, {y})')

# while: the condition decides, so it needs a bound
remaining = 3
while remaining > 0:
    remaining -= 1
    print(f'  steps left: {remaining}')

The practical rule is simple: reach for `for` when you have a sequence, and for `while` only when progress depends on a condition that changes inside the body. If nothing in the loop body changes the state the condition depends on, the loop cannot terminate - and that is a bug, not a design.

> **A range is a sequence, so a for loop over it is still safe**
>
> `for i in range(10)` has a definite length even though the loop looks condition-free. The length comes from the range object, not from a counter in the loop body.

## 7. Formal Theory

Iteration is a protocol, not a language feature. An object is iterable if it has an `__iter__` method returning an iterator, and an iterator has `__next__` returning the next value or raising `StopIteration` when it is done. A for loop is exactly that protocol driven for you, which is why any object you write can be used with for once it implements the two methods.

```text
for x in it:  ==  it = iter(it); loop { x = next(it) until StopIteration }
```

The exhaustion rule is what makes the for loop safe. The loop terminates because the iterator says so, not because a counter reached a limit - so an infinite sequence such as `itertools.count()` will happily loop forever unless the body itself breaks out.

In [ ]:
class Countdown:
    """A minimal iterator: iterable in itself, as Python expects."""

    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        self.current -= 1
        return self.current + 1


for value in Countdown(3):
    print(value, end=' ')
print()

| Loop | Stops when | Safe without a bound? | Typical use |
| --- | --- | --- | --- |
| `for` | the sequence is exhausted | yes | iterating data |
| `while` | the condition becomes false | **no** | waiting for a state |
| `for ... in range(n)` | n steps complete | yes | fixed counts |

## 8. Syntax

The canonical for loop names the sequence once and unpacks the element; the canonical while loop owns a counter it changes in the body.

In [ ]:
def drive_route(waypoints, max_speed_mps: float = 1.2) -> list[tuple]:
    """Return the waypoints that are within the speed limit."""
    visited: list[tuple] = []
    for index, point in enumerate(waypoints):
        if point[0] > max_speed_mps:
            continue
        visited.append((index, point))
    return visited


print(drive_route([(0.5, 1.0), (1.9, 2.0), (1.0, 3.0)]))

The anti-pattern - a while loop with no bound and no progress:

```text
battery = 48.0
while battery > 0:      # battery never changes -> infinite loop
    print('still driving')

while True:
    battery -= 0.5       # progress is visible in the body
    if battery <= 0:
        break
```

> **Every while loop needs a way out**
>
> If the body contains nothing that changes the condition, the loop is infinite. In a robot that is not a hang, it is a runaway - so every while loop here gets a bound, a break, or a state that provably moves.

## 9. Basic Examples

**Example 1 - iterating directly rather than by index.**

In [ ]:
readings = [0.4, 0.6, 0.5]

# for: the sequence supplies the element
for reading in readings:
    print(f'{reading:.1f}', end=' ')
print()

# manual index: the loop owns the bookkeeping
index = 0
while index < len(readings):
    print(f'{readings[index]:.1f}', end=' ')
    index += 1
print()

**Example 2 - the two loops side by side.**

In [ ]:
target = 7

steps = 0
while steps < target:
    steps += 1
print('while ->', steps)

count = 0
for _ in range(target):
    count += 1
print('for   ->', count)

## 10. Intermediate Examples

A waypoint follower is the canonical robotics loop: it walks a route, accumulates distance, and stops when the route is exhausted. Notice that the termination comes from the data, not from a counter.

In [ ]:
import math

WAYPOINTS = [(0.0, 0.0), (3.0, 0.0), (3.0, 4.0)]


def follow(points) -> dict:
    travelled = 0.0
    position = points[0]
    for target in points[1:]:
        travelled += math.dist(position, target)
        position = target
    return {'waypoints': len(points), 'distance_m': round(travelled, 3)}

## 11. Advanced Examples

Once you can write `__iter__` and `__next__`, you can make any object work with a for loop - which is how lazy sensor streams and infinite retry generators are built.

In [ ]:
class Readings:
    """Iterate a fixed list of readings without copying it."""

    def __init__(self, values):
        self._values = values
        self._index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self._index >= len(self._values):
            raise StopIteration
        value = self._values[self._index]
        self._index += 1
        return value


stream = Readings([0.1, 0.2, 0.3])
print('sum   :', round(sum(stream), 3))
print('drained:', round(sum(stream), 3))

That last line is the point of the protocol: once an iterator is exhausted it stays exhausted, so summing the same object twice gives zero. Iterating a list twice gives the same answer, because a list is re-iterable.

## 12. Code Walkthrough

Below is a mission-step planner: it walks a queue of steps, tracks progress, and refuses to run unbounded if a step keeps failing.

In [ ]:
"""planner.py - run mission steps with a bound on retries."""
from __future__ import annotations

MAX_ATTEMPTS = 3


def run_steps(steps, should_stop=lambda: False) -> dict:
    """Run each step, retrying a failure up to MAX_ATTEMPTS.

    Returns a report; never raises for a failing step, and stops
    early when the stop predicate becomes true.
    """
    completed: list[str] = []
    failed: list[str] = []
    for step in steps:
        if should_stop():
            break
        for attempt in range(1, MAX_ATTEMPTS + 1):
            if attempt == MAX_ATTEMPTS and not step.get('reliable'):
                failed.append(step['name'])
                break
            completed.append(step['name'])
            break
    return {'completed': completed, 'failed': failed,
            'stopped_early': len(completed) + len(failed) < len(steps)}

The outer loop is safe because the sequence is finite. The inner retry loop is a for over range, so it is bounded by construction - which is the pattern to copy whenever a retry needs a maximum. The stop predicate is checked at the top of each outer iteration so a mission can be aborted between steps.

## 13. Line-by-Line Explanation

1. MAX_ATTEMPTS is a module constant so the retry policy is visible at the top of the file rather than buried in a range call.
2. The outer for loop is finite because steps is a sequence - no condition involved, so it cannot run away.
3. The stop predicate is checked at the top of each iteration, so a mission can be aborted between steps rather than only between retries.
4. The inner loop is a for over range, which is bounded by construction; this is the safe way to express a retry.
5. The attempt counter is compared against MAX_ATTEMPTS explicitly, so the final failure is a decision rather than an accident.
6. A reliable step succeeds on the first attempt and breaks immediately, so the retry budget is only spent where it is needed.
7. stopped_early is derived by comparing counts, which keeps the report consistent with the lists rather than needing a separate flag.

## 14. Common Mistakes

**Mistake 1 - a while loop whose body does not change the condition.**

```text
battery = 48.0
while battery > 0:      # battery is never changed
    print('driving')

while battery > 0:      # progress is made each pass
    battery -= 0.5
```

**Mistake 2 - iterating by index when direct iteration will do.**

```text
for i in range(len(readings)):
    print(readings[i])

for reading in readings:
    print(reading)
```

**Mistake 3 - reusing an exhausted iterator.**

```text
stream = Readings([0.1, 0.2])
print(sum(stream))   # 0.3
print(sum(stream))   # 0.0 - it is drained

print(sum([0.1, 0.2]))   # a list can be re-iterated
```

**Mistake 4 - an unbounded retry loop.**

```text
while not sensor_ok():
    retry()

for attempt in range(MAX_ATTEMPTS):
    if sensor_ok():
        break
    retry()
```

## 15. Debugging Techniques

For a loop that runs the wrong number of times, the fastest diagnosis is to print the length of the sequence alongside the loop, and to check whether an iterator has already been consumed.

In [ ]:
readings = [0.4, 0.6, 0.5]
print('len  :', len(readings))
print('list :', list(readings))

iterator = iter(readings)
print('next :', next(iterator))
print('rest :', list(iterator))
print('empty:', list(iterator))

For a loop that never ends, instrument the condition rather than the body. Printing the values the condition depends on on every pass shows immediately whether it is changing.

In [ ]:
battery = 3.0
passes = 0
while battery > 0:
    battery -= 0.5
    passes += 1
    if passes > 10:
        print('bailed out: condition not converging')
        break
print('passes:', passes)

> **A safety bound is not a code smell**
>
> Adding a maximum iteration count to a loop you believe terminates costs nothing and turns a runaway into a diagnosable error.

## 16. Best Practices

- Reach for `for` when you have a sequence; it cannot run forever.
- Use `while` only when progress depends on a changing condition.
- Give every while loop a bound, a break, or a provably changing state.
- Iterate the element directly; index only when you need the position.
- Use `enumerate` rather than a manual counter.
- Do not reuse an iterator that may already be exhausted.
- Bound every retry loop with a fixed attempt count.
- Prefer a generator for a lazy sequence over building the whole list.

## 17. Performance Considerations

The for loop over a list is the fastest common loop in Python because the interpreter holds a direct reference to the list and never calls a method to fetch the next element. The equivalent manual index loop is measurably slower at the same asymptotics, and `for x in list` also avoids the bounds check that an index loop performs on every pass.

The real cost is in the body, not the loop. Appending to a list inside a loop is amortised O(1), but building a new string with `+=` on every iteration is quadratic because each concatenation copies what has accumulated. A generator avoids materialising the result at all, which matters when the sequence is long or infinite - and a sensor stream is often exactly that.

> **Measure the body**
>
> If a loop is slow, time the work inside it. Swapping a for loop for a while loop never makes anything faster.

## 18. Pythonic Approaches

Accumulating by index versus iterating the element.

In [ ]:
readings = [0.4, 0.9, 0.6]

# before: the loop owns the counter and the bound
total = 0.0
index = 0
while index < len(readings):
    total += readings[index]
    index += 1

# after: the sequence owns termination, the loop owns the work
total = 0.0
for reading in readings:
    total += reading

print('total:', round(total, 3))

## 19. Robotics Connection

A delivery cart follows a route, and the route is a sequence - so the outer loop is a for over waypoints. Inside, the cart may retry a motor command, and that retry is the case where a bound is mandatory.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

ROUTE = [(0.0, 0.0), (1.0, 0.0), (1.0, 1.0)]

robot = SimulatedRobot(name='cart-01', battery_wh=48.0)
legs = 0
for target in ROUTE[1:]:
    robot.move(distance_m=1.0, speed_mps=1.0)
    legs += 1
print(f'completed {legs} legs, battery {robot.status()["battery_pct"]:.1f}%')

## 20. Engineering Example

A bounded polling loop - the pattern a supervisor uses when it waits for a condition it cannot observe directly.

In [ ]:
def poll(check, attempts: int = 5, on_retry=None) -> bool:
    """Call check() up to `attempts` times; return True on the first success."""
    for attempt in range(1, attempts + 1):
        if check():
            return True
        if on_retry is not None:
            on_retry(attempt)
    return False

## 21. Guided Practice

1. Sum a list of readings three ways - a while loop with a counter, a for loop, and sum() - and confirm all three agree.
2. Write a for loop over a range that prints only the even indices.
3. Write a while loop with a counter that stops at 10, and explain what would make it infinite.
4. Define a class with __iter__ and __next__ that iterates a list backwards, then sum it twice to see what happens on the second pass.

In [ ]:
readings = [0.4, 0.9, 0.6]

total = 0.0
index = 0
while index < len(readings):
    total += readings[index]
    index += 1
print('while:', round(total, 3))

print('for  :', round(sum(readings), 3))
print('builtin:', round(sum(readings), 3))

count = 0
while count < 10:
    count += 1
print('bounded while reached', count)

for position in range(0, 6, 2):
    print('even position:', position)

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** What makes a for loop unable to run forever over a finite sequence?
- **Q1.** Why prefer `for x in items` over `for i in range(len(items))`?
- **Q1.** A telemetry stream never ends. Which structure is appropriate?
- **Q1.** Why must a motor-command retry be bounded?

## 23. Summary

A for loop does not know how many times it will run. It asks the object for one element at a time until the object says it is exhausted, which is why it is safe by construction - there is no counter to get wrong and no condition to forget. A while loop makes no such promise: it re-evaluates a condition, and it terminates only if something in the body changes the answer.

That asymmetry is the practical lesson. Reach for a for loop whenever you have a sequence, and for a while loop only when progress genuinely depends on a condition. Every while loop that reaches production needs a bound, a break, or a state that provably changes - and a retry loop should be a `for` over `range(MAX_ATTEMPTS)`, which is bounded for free.

Understanding iteration as a protocol rather than a language feature is what makes the rest of the course compose. Because `for` is exactly `iter()` plus `next()` until `StopIteration`, you can make your own objects work with it by implementing two methods - and you can see why a generator can be lazy and infinite, and why a consumed iterator stays consumed. For a telemetry pipeline this is not academic: sensor streams are long, sometimes endless, and a loop that cannot stop is a robot that cannot stop.

## 24. Key Takeaways

- A for loop is driven by the sequence and cannot run away.
- A while loop is driven by a condition and can run forever.
- Every while loop needs a bound, a break, or visible progress.
- Iterate the element directly; use enumerate when you need the index.
- Implement __iter__ and __next__ to make your own objects loopable.
- An exhausted iterator stays exhausted - do not reuse it.
- Bound every retry loop with a fixed attempt count.
- A for over range(n) is the safe way to express a fixed count.

## 25. Further Exploration

- The for statement](https://docs.python.org/3/reference/compound_stmts.html#the-for-statement)
- Data model - the iterator protocol](https://docs.python.org/3/reference/datamodel.html#object.__iter__)
- Writing classes - iterators](https://docs.python.org/3/tutorial/classes.html#iterators)
- itertools - fast, memory-efficient iterators](https://docs.python.org/3/library/itertools.html)
- PEP 234 - the iterator protocol](https://peps.python.org/pep-0234/)